# Day 2 · Hands-on 5 — Counts: Poisson and negative binomial models
**About 25 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)


Counts (lymph nodes, lesions, cells, sequencing reads) are whole numbers ≥ 0, often skewed, and their spread grows with their mean. **Poisson** and **negative binomial (NB)** regression model them directly. The NB model is also the engine behind RNA-seq differential expression.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.


**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 appear only for the simulated dataset, where we know the true answer.

In [ ]:
# @title ▶️ Setup: run this cell first (no changes needed) { display-mode: "form" }
# Every analysis below uses ROLES (outcome, group, ...), never hard-coded column names.
# To add a dataset: write a loader function and add one entry to DATASETS.
%pip install -q lifelines
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from lifelines.datasets import load_gbsg2, load_lung

def _gbsg2():
    d = load_gbsg2()
    for c in ["tgrade", "menostat", "horTh"]:
        d[c] = d[c].astype(str)
    d["nodes4"] = (d["pnodes"] >= 4).astype(int)
    return d

def _lung():
    d = load_lung()
    d.columns = [c.replace(".", "_") for c in d.columns]
    d = d.dropna(subset=["ph_ecog"])
    d["ecog"] = d["ph_ecog"].map({0: "0", 1: "1", 2: "2+", 3: "2+"})
    d["sex"] = d["sex"].map({1: "male", 2: "female"})
    d["lost_weight"] = np.where(d["wt_loss"].isna(), np.nan, (d["wt_loss"] > 0).astype(float))
    return d

def _simulated(seed=2026, n=300):
    rng = np.random.default_rng(seed)
    genotype = rng.choice(["AA", "AB", "BB"], size=n, p=[0.4, 0.4, 0.2])
    sex = rng.choice(["F", "M"], size=n)
    age = rng.uniform(30, 80, n) + np.where(genotype == "BB", 8, 0)          # BB carriers are older
    g_eff = pd.Series({"AA": 0, "AB": 3, "BB": 8})[genotype].to_numpy()
    slope = np.where(sex == "M", 0.7, 0.4)                                    # interaction: steeper in men
    biomarker = 50 + slope * (age - 50) + g_eff + rng.normal(0, 6, n)
    logit = -1 + 0.05 * (age - 50) + pd.Series({"AA": 0, "AB": 0.4, "BB": 1.0})[genotype].to_numpy()
    disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    mu = np.exp(0.5 + 0.02 * (age - 50) + pd.Series({"AA": 0, "AB": 0.3, "BB": 0.7})[genotype].to_numpy())
    alpha = 0.8
    lesions = rng.negative_binomial(1 / alpha, 1 / (1 + alpha * mu))
    return pd.DataFrame(dict(biomarker=biomarker.round(1), genotype=genotype, sex=sex, age=age.round(1),
                             disease=disease, lesions=lesions))

def _my_data():
    # Upload a CSV to Colab (folder icon on the left) and edit the roles in DATASETS["my_data"]
    return pd.read_csv("my_data.csv")

DATASETS = {
    "gbsg2": dict(load=_gbsg2,
        about="GBSG2 breast cancer trial: 686 patients",
        outcome="tsize", outcome_label="Tumour size (mm)",
        group="tgrade", group_label="Tumour grade",
        x="pnodes", x_label="Positive lymph nodes",
        group2="menostat", group2_label="Menopausal status",
        binary="nodes4", binary_label="4 or more positive lymph nodes",
        binary_x="tsize", binary_group="tgrade",
        count="pnodes", count_label="Positive lymph nodes", count_x="tsize", count_group="tgrade"),
    "lung": dict(load=_lung,
        about="NCCTG advanced lung cancer study: 227 patients",
        outcome="pat_karno", outcome_label="Patient-rated Karnofsky score",
        group="ecog", group_label="ECOG performance status",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="lost_weight", binary_label="Lost weight in last 6 months",
        binary_x="age", binary_group="ecog",
        count=None),
    "simulated": dict(load=_simulated,
        about="Simulated biomarker study: 300 people, true effects known",
        outcome="biomarker", outcome_label="Biomarker level",
        group="genotype", group_label="Genotype",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="disease", binary_label="Disease present",
        binary_x="age", binary_group="genotype",
        count="lesions", count_label="Number of lesions", count_x="age", count_group="genotype",
        truth={
            "group": "AB vs AA = +3, BB vs AA = +8 (adjusted for age). BB carriers are also ~8 years older.",
            "x": "age slope = 0.4 per year in women, 0.7 in men",
            "interaction": "difference in age slopes (M vs F) = +0.3 per year",
            "binary": "odds ratio per year of age = 1.05; AB vs AA = 1.49; BB vs AA = 2.72",
            "count": "rate ratio per year of age = 1.02; AB vs AA = 1.35; BB vs AA = 2.01; overdispersed (alpha = 0.8)",
        }),
    "my_data": dict(load=_my_data,
        about="Your own CSV: edit these roles to match your column names",
        outcome="my_outcome", outcome_label="My outcome",
        group="my_group", group_label="My group",
        x="my_numeric_predictor", x_label="My numeric predictor",
        group2="my_two_level_group", group2_label="My two-level group",
        binary="my_yes_no_outcome", binary_label="My yes/no outcome (coded 0/1)",
        binary_x="my_numeric_predictor", binary_group="my_group",
        count=None),
}

def load_dataset(name):
    R = dict(DATASETS[name])
    d = R.pop("load")()
    for role in ["group", "group2", "binary_group", "count_group"]:
        if R.get(role):
            d[R[role]] = d[R[role]].astype(str)
    R["levels"] = sorted(d[R["group"]].dropna().unique())
    R.setdefault("truth", {})
    print(f"✅ {R['about']}  ({len(d)} rows)")
    print(f"   outcome: {R['outcome']} ({R['outcome_label']})   group: {R['group']} {R['levels']}")
    print(f"   numeric predictor: {R['x']}   second group: {R['group2']}")
    return d, R

def cat(var, ref):
    """Categorical term with an explicit reference level."""
    return f"C({var}, Treatment(reference='{ref}'))"

def nice(term):
    """Turn statsmodels term names into readable labels."""
    m = re.fullmatch(r"C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\](.*)", term)
    if m:
        var, ref, lvl, rest = m.groups()
        label = f"{var}: {lvl} vs {ref}"
        return label + (f" × {rest.lstrip(':')}" if rest else "")
    m = re.fullmatch(r"(\w+):C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\]", term)
    if m:
        x, var, ref, lvl = m.groups()
        return f"{x} × {var}: {lvl} vs {ref}"
    return term

def tidy(fit, exponentiate=False):
    """Coefficient table: estimate, 95% CI, p-value."""
    ci = fit.conf_int()
    t = pd.DataFrame({"estimate": fit.params, "CI low": ci[0], "CI high": ci[1], "p-value": fit.pvalues})
    t = t.drop(index=[i for i in t.index if i == "alpha"], errors="ignore")
    if exponentiate:
        t[["estimate", "CI low", "CI high"]] = np.exp(t[["estimate", "CI low", "CI high"]])
    t.index = [nice(i) for i in t.index]
    return t.round({"estimate": 3, "CI low": 3, "CI high": 3}).assign(**{"p-value": t["p-value"].map(lambda p: f"{p:.2g}")})

def show_truth(R, key):
    if key in R["truth"]:
        print(f"🎯 Truth used to simulate: {R['truth'][key]}")

print("✅ Setup complete")

# ---- RNA-seq datasets: a counts table (genes × samples) + a sample table ----
AIRWAY = "https://raw.githubusercontent.com/bioboot/bggn213_W19/master/class-material/airway_scaledcounts.csv"
AIRWAY_META = "https://raw.githubusercontent.com/bioboot/bggn213_W19/master/class-material/airway_metadata.csv"

def _airway():
    counts = pd.read_csv(AIRWAY, index_col=0)
    meta = pd.read_csv(AIRWAY_META, index_col=0)
    return counts, meta, None

def _sim_rnaseq(seed=7, n_genes=8000, n_donors=4, frac_de=0.10):
    rng = np.random.default_rng(seed)
    base = rng.lognormal(5, 1.8, n_genes)                       # typical expression of each gene
    disp = 0.03 + 1.5 / np.sqrt(base)                           # biological overdispersion
    is_de = rng.random(n_genes) < frac_de
    lfc = np.where(is_de, rng.choice([-1, 1], n_genes) * rng.uniform(0.5, 2.5, n_genes), 0)
    donor = rng.normal(0, 0.5, (n_donors, n_genes))             # donor-to-donor differences
    cols, rows = {}, []
    for dnr in range(n_donors):
        for cond, k in [("control", 0), ("treated", 1)]:
            mu = base * np.exp(donor[dnr]) * 2 ** (lfc * k) * rng.uniform(0.7, 1.3)
            name = f"donor{dnr + 1}_{cond}"
            cols[name] = rng.negative_binomial(1 / disp, 1 / (1 + disp * mu))
            rows.append({"sample": name, "condition": cond, "donor": f"donor{dnr + 1}"})
    genes = [f"gene{i:05d}" for i in range(n_genes)]
    return (pd.DataFrame(cols, index=genes), pd.DataFrame(rows).set_index("sample"),
            pd.Series(is_de, index=genes))

def _my_rnaseq():
    # Upload my_counts.csv (genes as rows, samples as columns) and my_metadata.csv (one row per sample)
    return pd.read_csv("my_counts.csv", index_col=0), pd.read_csv("my_metadata.csv", index_col=0), None

RNASEQ = {
    "airway": dict(load=_airway, about="Airway smooth muscle cells ± dexamethasone (GSE52778): 4 cell lines × 2",
                   condition="dex", reference="control", treated="treated", block="celltype"),
    "simulated": dict(load=_sim_rnaseq, about="Simulated: 8,000 genes, 4 donors × 2 conditions, 10% truly DE",
                      condition="condition", reference="control", treated="treated", block="donor"),
    "my_data": dict(load=_my_rnaseq, about="Your own counts: edit these roles",
                    condition="my_condition", reference="my_control_level", treated="my_treated_level", block=None),
}

def load_rnaseq(name):
    S = dict(RNASEQ[name])
    counts, meta, truth = S.pop("load")()
    counts = counts.round().astype(int)
    meta = meta.loc[counts.columns]
    S["truth"] = truth
    print(f"✅ {S['about']}")
    print(f"   {counts.shape[0]} genes × {counts.shape[1]} samples;  condition column: '{S['condition']}';  block: {S['block']}")
    return counts, meta, S

## Choose a dataset
Everything in this notebook is written in terms of **roles** (an outcome, a group, a numeric predictor), so the same cells work on any dataset. Start with the one your facilitator names, then try `simulated`: it was generated with known effects, so you can check whether the model recovers the truth.

In [ ]:
# 🔧 CHANGE THIS: which dataset to use (all cells below adapt automatically)
dataset = "gbsg2"  # @param ["gbsg2", "lung", "simulated", "my_data"]

df, R = load_dataset(dataset)
df.head()

## A. What do the counts look like?
### 🧭 Concept
A Poisson distribution has **variance = mean**. Real biological counts usually have variance > mean: that's **overdispersion**.

✍️ **Predict first:** for your count, will the variance be close to the mean, or much larger?

In [ ]:
# 🔧 CHANGE THIS: count variable (any column of whole numbers ≥ 0)
count_var = R["count"]

if count_var is None:
    print("This dataset has no count outcome. Choose 'gbsg2' or 'simulated' in the dataset cell.")
else:
    c = df[count_var].dropna()
    plt.figure(figsize=(6, 3.5))
    plt.hist(c, bins=np.arange(c.max() + 2) - 0.5, color="gray", edgecolor="white")
    plt.xlabel(count_var); plt.ylabel("Number of rows"); plt.show()
    print(f"Mean = {c.mean():.2f}   Variance = {c.var():.2f}   Variance / mean = {c.var() / c.mean():.1f}")
    print(f"Proportion of zeros = {(c == 0).mean():.1%}")

## B. Poisson vs negative binomial
### 🧭 Concept
Both models give **rate ratios** (exponentiated coefficients): how many times higher the expected count is per unit of predictor or relative to the reference group. The NB model adds a dispersion parameter, α, that lets the variance be larger than the mean.

Fit each model and compare the confidence intervals.

In [ ]:
# 🔧 CHANGE THIS: "poisson" or "negbin"
model = "poisson"

assert R["count"], "This dataset has no count outcome: choose 'gbsg2' or 'simulated' in the dataset cell."
yc, x, g = R["count"], R["count_x"], R["count_group"]
d = df.dropna(subset=[yc, x, g])
ref = sorted(d[g].unique())[0]
formula = f"{yc} ~ {x} + {cat(g, ref)}"
if model == "poisson":
    count_fit = smf.poisson(formula, data=d).fit(disp=0)
    pearson = np.sum((d[yc] - count_fit.predict()) ** 2 / count_fit.predict()) / count_fit.df_resid
    print(f"Dispersion check (Pearson χ² / df): {pearson:.1f}   (about 1 if Poisson fits; much larger = overdispersed)")
else:
    count_fit = smf.negativebinomial(formula, data=d).fit(disp=0)
    print(f"Estimated dispersion α = {count_fit.params['alpha']:.2f}   (variance = mean + α × mean²)")
print("\nRate ratios with 95% confidence intervals:")
display(tidy(count_fit, exponentiate=True))
show_truth(R, "count")

## C. Which model reproduces the data?
A good model should predict the observed **shape**: how many zeros, how many very large counts. If the data contain **no zeros at all**, ask why: how were the subjects selected? (Neither model knows about that.)

In [ ]:
# 🔧 CHANGE THIS: a "large" count threshold
large = 15

mu = count_fit.predict()
if model == "poisson":
    p_zero, p_large = np.mean(stats.poisson.pmf(0, mu)), np.mean(stats.poisson.sf(large - 1, mu))
else:
    a = count_fit.params["alpha"]; n_ = 1 / a; p_ = n_ / (n_ + mu)
    p_zero, p_large = np.mean(stats.nbinom.pmf(0, n_, p_)), np.mean(stats.nbinom.sf(large - 1, n_, p_))
print(f"Model: {model}")
print(f"   zeros:        observed {np.mean(d[yc] == 0):.1%}   predicted {p_zero:.1%}")
print(f"   ≥ {large}:  observed {np.mean(d[yc] >= large):.1%}   predicted {p_large:.1%}")

### ✅ Concept check
The Poisson and NB rate ratios are similar, but their confidence intervals differ. Which one should you trust, and why does it matter?

<details><summary>Show answer</summary>

If the data are overdispersed, the Poisson model assumes less noise than there is, so its confidence intervals are **too narrow** and its p-values **too small**: false positives. The NB model accounts for the extra variation.
</details>

## D. Bridge to RNA-seq: every gene is a count
### 🧭 Concept
In RNA-seq each gene's read count across replicate samples is a count. The plot shows, for every gene, its mean and variance across the **control** samples. The dashed line is variance = mean (Poisson).

In [ ]:
# 🔧 CHANGE THIS: which RNA-seq dataset to use
rnaseq_dataset = "airway"  # @param ["airway", "simulated", "my_data"]

counts, meta, S = load_rnaseq(rnaseq_dataset)
meta

In [ ]:
# 🔧 CHANGE THIS: minimum mean count for a gene to be plotted
min_mean = 1

ctrl = counts.loc[:, meta[S["condition"]] == S["reference"]]
norm = ctrl / (ctrl.sum() / ctrl.sum().mean())               # simple library-size scaling
m, v = norm.mean(axis=1), norm.var(axis=1)
keep = m >= min_mean
plt.figure(figsize=(6, 4.5))
plt.loglog(m[keep], v[keep], ".", alpha=0.15, markersize=3)
xs = np.logspace(np.log10(min_mean), np.log10(m.max()), 50)
plt.loglog(xs, xs, "k--", label="Poisson: variance = mean")
alpha_hat = np.median(((v - m) / m ** 2)[m > 100])
plt.loglog(xs, xs + alpha_hat * xs ** 2, "r-", label=f"NB: variance = mean + {alpha_hat:.2f}·mean²")
plt.xlabel("Mean count"); plt.ylabel("Variance"); plt.legend(); plt.title("Each dot is one gene"); plt.show()

## 🤖 Go deeper with AI
Three ways to keep learning about count models: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain overdispersion with an example from counting cells or colonies, and why it makes Poisson p-values too small. Then give me 2 datasets (mean and variance only) and ask: Poisson or negative binomial?*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Simulate 2,000 two-group experiments with overdispersed counts (negative binomial, α = 0.8) and no true difference. Compare how often a Poisson and a negative binomial model give p < 0.05.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *I count [cells / colonies / lesions] in my experiments: [describe how]. Which model fits, and do I need an offset, for example for the area imaged or the number of fields?*

✔ **Check one claim yourself:** The Poisson false-positive rate should be well above 5%; the negative binomial close to 5%.

## 💬 Discuss with your partner
- Which counts in your own work (cells, colonies, events, reads) are likely overdispersed?
- In section D, at low mean counts the dots sit near the Poisson line; at high counts they don't. What kind of variation dominates in each regime?
- Why can't you just log-transform counts and use a linear model? (When might that be acceptable?)